# DuckLake Conference Live Demo

Assumes the table already exists. Run `streaming_demo` first to populate it with
millions of rows. This notebook walks through: connect + explore, ACID transactions,
time travel, schema evolution + CDC, MERGE/upsert, and maintenance.

In [ ]:
import sys
from pathlib import Path

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

_REPO = Path.cwd()
if _REPO.name == "notebooks":
    _REPO = _REPO.parent
if str(_REPO / "src") not in sys.path:
    sys.path.insert(0, str(_REPO / "src"))

from ducklake_playground import DuckLakeEngine, load_config

config = load_config(_REPO / "config.yaml")
engine = DuckLakeEngine()
engine.setup(config, config.default_storage_mode)
con = engine.connection
catalog = engine.catalog_name
TABLE = "demo_table"
fq = f"{catalog}.main.{TABLE}"
print(f"Connected to {catalog} | table: {fq}")

## 1. Explore the Table

In [ ]:
con.execute(f"DESCRIBE {fq}").pl()

In [ ]:
con.execute(f"""
    SELECT COUNT(*)                   AS total_rows,
           MIN(event_date)            AS first_date,
           MAX(event_date)            AS last_date,
           COUNT(DISTINCT event_date) AS partitions
    FROM {fq}
""").pl()

In [ ]:
con.execute(f"""
    SELECT varchar_col,
           COUNT(*)         AS cnt,
           SUM(int64_col)   AS total,
           AVG(float64_col) AS avg_val
    FROM {fq}
    WHERE event_date BETWEEN DATE '2024-01-10' AND DATE '2024-01-15'
    GROUP BY varchar_col
    ORDER BY cnt DESC
    LIMIT 10
""").pl()

In [ ]:
# Prove partition pruning to the audience
con.execute(f"""
    EXPLAIN ANALYZE
    SELECT varchar_col, COUNT(*) AS cnt
    FROM {fq}
    WHERE event_date = DATE '2024-01-15'
    GROUP BY varchar_col
""").pl()

## 2. ACID Transactions

Multi-statement transaction: INSERT + UPDATE land atomically in one DuckLake snapshot.

In [ ]:
pre_count = con.execute(f"SELECT COUNT(*) FROM {fq}").fetchone()[0]
pre_snapshot = con.execute(
    f"SELECT MAX(snapshot_id) FROM {catalog}.snapshots()"
).fetchone()[0]
print(f"Before transaction: {pre_count:,} rows | snapshot v{pre_snapshot}")

In [ ]:
con.execute("BEGIN TRANSACTION")
try:
    # Insert two new rows
    con.execute(f"""
        INSERT INTO {fq} (id, event_date, int64_col, float64_col, varchar_col)
        VALUES
            (900000001, DATE '2024-01-15', 1499, 99.95, 'value_042'),
            (900000002, DATE '2024-01-15', 49,   19.99, 'value_007')
    """)
    # Update one of them (10% discount)
    con.execute(f"""
        UPDATE {fq}
        SET float64_col = float64_col * 0.9
        WHERE id = 900000001
    """)
    con.execute("COMMIT")
    print("COMMITTED")
except Exception as exc:
    con.execute("ROLLBACK")
    print(f"ROLLED BACK: {exc}")

post_count = con.execute(f"SELECT COUNT(*) FROM {fq}").fetchone()[0]
post_snapshot = con.execute(
    f"SELECT MAX(snapshot_id) FROM {catalog}.snapshots()"
).fetchone()[0]
print(f"After transaction: {post_count:,} rows (v{post_snapshot}, +{post_count - pre_count})")

In [ ]:
# Verify: both the INSERT and UPDATE landed
con.execute(f"""
    SELECT id, event_date, int64_col, float64_col, varchar_col
    FROM {fq}
    WHERE id IN (900000001, 900000002)
    ORDER BY id
""").pl()

## 3. Time Travel & Snapshots

Each write creates a new immutable snapshot. Query any historical version.

In [ ]:
con.execute(f"""
    SELECT snapshot_id, snapshot_time, changes
    FROM {catalog}.snapshots()
    ORDER BY snapshot_id DESC
    LIMIT 10
""").pl()

In [ ]:
print(f"Before-tx snapshot: v{pre_snapshot} | After-tx snapshot: v{post_snapshot}")

In [ ]:
# Query the table AS OF the previous snapshot (before our transaction)
con.execute(f"""
    SELECT COUNT(*) AS row_count_before_tx
    FROM {fq} AT (VERSION => {pre_snapshot})
""").pl()

In [ ]:
# Prove the inserted rows did NOT exist in the previous version
con.execute(f"""
    SELECT id, event_date, float64_col
    FROM {fq} AT (VERSION => {pre_snapshot})
    WHERE id IN (900000001, 900000002)
""").pl()

## 4. Change Data Feed (CDC)

What changed between two snapshots? No Kafka or external tooling required.

In [ ]:
con.execute(f"""
    SELECT *
    FROM {catalog}.table_changes('{TABLE}', {pre_snapshot}, {post_snapshot})
    ORDER BY change_type, id
    LIMIT 20
""").pl()

## 5. Schema Evolution

ADD / RENAME / DROP columns without rewriting any Parquet files.

In [ ]:
# Add a new column (metadata-only, no Parquet rewrite)
con.execute(f"ALTER TABLE {fq} ADD COLUMN priority VARCHAR DEFAULT 'normal'")
print("Column 'priority' added. Existing Parquet files untouched.")

In [ ]:
# Verify: new column appears, old rows have the default
con.execute(f"""
    SELECT id, event_date, varchar_col, priority
    FROM {fq}
    WHERE id IN (900000001, 900000002, 1, 2, 3)
    ORDER BY id
    LIMIT 5
""").pl()

In [ ]:
# Rename column (metadata-only, zero Parquet I/O)
con.execute(f"ALTER TABLE {fq} RENAME COLUMN priority TO urgency")
print("Renamed 'priority' to 'urgency'. Zero file I/O.")

In [ ]:
# Drop column to restore the table for the next demo
con.execute(f"ALTER TABLE {fq} DROP COLUMN urgency")
print("Dropped 'urgency'. Schema restored.")

## 6. MERGE / Upsert

Atomic upsert: update existing rows + insert new ones in a single snapshot.

In [ ]:
con.execute(f"""
    MERGE INTO {fq} AS target
    USING (
        VALUES
            (900000001, DATE '2024-01-15', CAST(9999 AS BIGINT),
             CAST(42.0 AS DOUBLE), 'value_042'),
            (999999999, DATE '2024-01-20', CAST(7777 AS BIGINT),
             CAST(55.5 AS DOUBLE), 'value_001')
    ) AS source(id, event_date, int64_col, float64_col, varchar_col)
    ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET int64_col = source.int64_col,
                   float64_col = source.float64_col
    WHEN NOT MATCHED THEN
        INSERT (id, event_date, int64_col, float64_col, varchar_col)
        VALUES (source.id, source.event_date, source.int64_col,
                source.float64_col, source.varchar_col)
""")
print("MERGE complete: id=900000001 updated, id=999999999 inserted.")

In [ ]:
# Verify MERGE results
con.execute(f"""
    SELECT id, event_date, int64_col, float64_col, varchar_col
    FROM {fq}
    WHERE id IN (900000001, 999999999)
    ORDER BY id
""").pl()

## 7. Maintenance

DuckLake **does** require maintenance: file compaction, snapshot expiry, and cleanup.

In [ ]:
# File statistics BEFORE compaction
con.execute(f"""
    SELECT COUNT(*)                                    AS total_files,
           ROUND(SUM(data_file_size_bytes) / 1e6, 2)  AS total_mb,
           ROUND(AVG(data_file_size_bytes) / 1e6, 2)  AS avg_file_mb,
           ROUND(MIN(data_file_size_bytes) / 1e6, 2)  AS min_file_mb,
           ROUND(MAX(data_file_size_bytes) / 1e6, 2)  AS max_file_mb
    FROM ducklake_list_files('{catalog}', '{TABLE}')
""").pl()

In [ ]:
# Compact small files into larger ones
con.execute(f"CALL ducklake_merge_adjacent_files('{catalog}')")
print("ducklake_merge_adjacent_files complete.")

In [ ]:
# File statistics AFTER compaction
con.execute(f"""
    SELECT COUNT(*)                                    AS total_files,
           ROUND(SUM(data_file_size_bytes) / 1e6, 2)  AS total_mb,
           ROUND(AVG(data_file_size_bytes) / 1e6, 2)  AS avg_file_mb
    FROM ducklake_list_files('{catalog}', '{TABLE}')
""").pl()

In [ ]:
# Expire old snapshots (aggressive for demo: 1 day)
con.execute(f"CALL ducklake_expire_snapshots('{catalog}', older_than => now() - INTERVAL '1 day')")
print("ducklake_expire_snapshots complete.")

# Clean up orphaned data files
con.execute(f"CALL ducklake_cleanup_old_files('{catalog}', cleanup_all => true)")
print("ducklake_cleanup_old_files complete.")

In [ ]:
# Final snapshot list
con.execute(f"""
    SELECT snapshot_id, snapshot_time, changes
    FROM {catalog}.snapshots()
    ORDER BY snapshot_id DESC
    LIMIT 10
""").pl()

## Cleanup

Remove demo artifacts to restore the table to its pre-demo state.

In [ ]:
# Uncomment and run after the presentation:
# con.execute(f"DELETE FROM {fq} WHERE id IN (900000001, 900000002, 999999999)")
# print("Demo rows removed.")

In [ ]:
# engine.close()